Environment: `velocity`

In [1]:
import scanpy
import anndata
import matplotlib
from matplotlib import pyplot
import hdf5plugin
import numpy
import scvelo
import seaborn
import pandas
import warnings
import cellrank
import gseapy

In [2]:
# Read input file
working_directory = "RNA Sequencing Data/"

# adata = scanpy.read_h5ad(working_directory + "/velocity_sdevelo.h5ad")

# Or read saved anndata objects
working_directory = "RNA Sequencing Data/"
base_name = "cellrank_version_5"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name}_anndata.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name}_driver_genes.csv",
    index_col=0
)
combined_driver_df = pandas.read_csv(
    working_directory+f"{base_name}_driver_genes_combined.csv",
    index_col=0
)

In [3]:
# Read ChIP-seq targets
base_name = "ChIP Sequencing Data/chipseq_targets"

hic2_target_scores = pandas.read_csv(f"{base_name}_hic2_scores.csv", index_col=0)
klf4_hic2_target_scores = pandas.read_csv(f"{base_name}_klf4_hic2_scores.csv", index_col=0)
klf4_control_target_scores = pandas.read_csv(f"{base_name}_klf4_control_scores.csv", index_col=0)

hic2_targets = numpy.loadtxt(f"{base_name}_hic2_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_hic2_targets = numpy.loadtxt(f"{base_name}_klf4_hic2_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_control_targets = numpy.loadtxt(f"{base_name}_klf4_control_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_targets = numpy.intersect1d(klf4_hic2_targets, klf4_control_targets)
shared_targets = numpy.intersect1d(hic2_targets, klf4_targets)

In [9]:
# Load bulk differential expression results

bulk_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_combined.csv", index_col=0)
bulk_metadata = pandas.read_csv("RNA Sequencing Data/bulk_sequencing_metadata.csv", index_col=0)
samples = bulk_metadata.index.to_list()
reprogramming_columns = bulk_de.columns[bulk_de.columns.str.contains("Reprogramming")]
reprogramming_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_reprogramming.csv", index_col=0)
mef_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_mef.csv", index_col=0)
mesc_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_mesc.csv", index_col=0)
ipsc_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_ipsc.csv", index_col=0)

In [ ]:
# Variable genes and Cellrank drivers
variable_genes = adata.var_names.to_numpy()
variable_in_bulk = numpy.intersect1d(variable_genes, bulk_de.index.to_numpy())
stem_cell_drivers = driver_df.sort_values("Day 12 Control_corr", ascending=True).head(100).index.to_numpy()
dead_end_drivers = driver_df.sort_values("Day 12 Control_corr", ascending=False).head(100).index.to_numpy()
differentially_expressed_genes = reprogramming_de.loc[variable_in_bulk].query("padj < 0.05").index.to_numpy()

In [18]:
# Load list of transcription factors

tf_df = pandas.read_csv("ChIP Sequencing Data/Mus_musculus_TF.txt", sep="\t")
tf_array = tf_df["Symbol"].to_numpy()

# All target transcription factors
target_tfs = numpy.intersect1d(shared_targets, tf_array)

# Target transcription factors for which Hic2 changes expression significantly
de_target_tfs = numpy.intersect1d(target_tfs, differentially_expressed_genes)

de_dead_end_drivers = numpy.intersect1d(de_target_tfs, dead_end_drivers)
de_stem_cell_drivers = numpy.intersect1d(de_target_tfs, stem_cell_drivers)

In [21]:
genes_of_interest = de_dead_end_drivers.tolist() + de_stem_cell_drivers.tolist() + ["Klf4", "Myc", "Hic2"]

In [4]:
# Show all possible libraries
names = gseapy.get_library_name(organism="Mouse")
print(names)

['ARCHS4_Cell-lines', 'ARCHS4_IDG_Coexp', 'ARCHS4_Kinases_Coexp', 'ARCHS4_TFs_Coexp', 'ARCHS4_Tissues', 'Achilles_fitness_decrease', 'Achilles_fitness_increase', 'Aging_Perturbations_from_GEO_down', 'Aging_Perturbations_from_GEO_up', 'Allen_Brain_Atlas_10x_scRNA_2021', 'Allen_Brain_Atlas_down', 'Allen_Brain_Atlas_up', 'Azimuth_2023', 'Azimuth_Cell_Types_2021', 'BioCarta_2013', 'BioCarta_2015', 'BioCarta_2016', 'BioPlanet_2019', 'BioPlex_2017', 'CCLE_Proteomics_2020', 'CM4AI_U2OS_Protein_Localization_Assemblies', 'COMPARTMENTS_Curated_2025', 'COMPARTMENTS_Experimental_2025', 'CORUM', 'COVID-19_Related_Gene_Sets', 'COVID-19_Related_Gene_Sets_2021', 'Cancer_Cell_Line_Encyclopedia', 'Carcinogenome', 'CellMarker_2024', 'CellMarker_Augmented_2021', 'ChEA_2013', 'ChEA_2015', 'ChEA_2016', 'ChEA_2022', 'Chromosome_Location', 'Chromosome_Location_hg19', 'ClinVar_2019', 'ClinVar_2025', 'DGIdb_Drug_Targets_2024', 'DSigDB', 'Data_Acquisition_Method_Most_Popular_Genes', 'DepMap_CRISPR_GeneDependency

# Gene set enrichment for Cellrank drivers

## GO pathway

In [5]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Biological_Process_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-02-02 11:08:53,006 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [6]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Epidermis Development (GO:0008544),0.65148,1.832768,0.002571,0.4625,0.673351
1,Regulation of Trans-Synaptic Signaling (GO:009...,0.794156,1.804547,0.001188,0.593,0.504901
2,Regulation of Glycolytic Process (GO:0006110),0.816355,1.756093,0.00489,0.802,0.66826
3,Proteolysis Involved in Protein Catabolic Proc...,0.743789,1.755604,0.006105,0.803,0.504452
7,Antigen Processing and Presentation of Exogeno...,0.858342,1.711653,0.010753,0.935,0.505029
5,Antigen Processing and Presentation of Exogeno...,0.858342,1.711653,0.010753,0.935,0.505029
6,Antigen Processing and Presentation of Peptide...,0.858342,1.711653,0.010753,0.935,0.505029
10,Negative Regulation of Myeloid Leukocyte Diffe...,0.761694,1.639877,0.025862,0.9925,0.978021
11,Positive Regulation of Purine Nucleotide Catab...,0.871029,1.638283,0.009423,0.9925,0.795758
12,Positive Regulation of Glycolytic Process (GO:...,0.871029,1.638283,0.009423,0.9925,0.795758


In [7]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
4,Chromatin Organization (GO:0006325),-1.740148,0.003165,0.508,0.95249
8,Protein Polyubiquitination (GO:0000209),-1.69159,0.002494,0.7605,1.0
9,DNA Metabolic Process (GO:0006259),-1.652498,0.011735,0.9135,1.0
13,Male Meiotic Nuclear Division (GO:0007140),-1.636868,0.000833,0.9525,1.0
15,Transcription by RNA Polymerase II (GO:0006366),-1.626133,0.004149,0.9685,1.0
16,Transcription Initiation-Coupled Chromatin Rem...,-1.600658,0.006071,0.9885,1.0
17,Regulation of Viral Genome Replication (GO:004...,-1.594761,0.017964,0.991,1.0
18,Regulation of Insulin Secretion (GO:0050796),-1.592851,0.008584,0.992,1.0
24,Visual System Development (GO:0150063),-1.552392,0.006809,0.999,1.0
26,Negative Regulation of Bone Remodeling (GO:004...,-1.538783,0.006843,1.0,1.0


## GO molecular function

In [8]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Molecular_Function_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-02-02 11:09:29,168 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [9]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Chemokine Receptor Binding (GO:0042379),0.785248,1.921597,0.0,0.0225,0.028389
1,Cysteine-Type Endopeptidase Activity (GO:0004197),0.679228,1.680922,0.014252,0.457,0.374682
3,Exopeptidase Activity (GO:0008238),0.800233,1.647327,0.023284,0.577,0.36138
4,Chemokine Activity (GO:0008009),0.710391,1.628463,0.024643,0.6415,0.330893
6,Cysteine-Type Peptidase Activity (GO:0008234),0.613392,1.593236,0.02439,0.7675,0.376664
7,Carboxypeptidase Activity (GO:0004180),0.763355,1.589139,0.029817,0.781,0.326742
11,Monoatomic Cation Channel Activity (GO:0005261),0.663853,1.555057,0.040342,0.8685,0.377016
12,GTP Binding (GO:0005525),0.534418,1.522445,0.021978,0.933,0.429786
13,Phosphatase Binding (GO:0019902),0.674406,1.491764,0.068075,0.9625,0.476126
14,Guanyl Ribonucleotide Binding (GO:0032561),0.489552,1.46521,0.035088,0.9835,0.516413


In [10]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,mRNA Binding (GO:0003729),-1.662355,0.006809,0.282,0.370934
5,Iron Ion Binding (GO:0005506),-1.598965,0.004992,0.549,0.462834
8,2-Oxoglutarate-Dependent Dioxygenase Activity ...,-1.5827,0.007765,0.616,0.371886
9,Methylated Histone Binding (GO:0035064),-1.558144,0.009821,0.73,0.304556
10,Methylation-Dependent Protein Binding (GO:0140...,-1.558144,0.009821,0.73,0.304556
16,Protein Kinase C Binding (GO:0005080),-1.430195,0.064378,0.9865,0.924556
17,Guanyl-Nucleotide Exchange Factor Activity (GO...,-1.423662,0.079723,0.9915,0.841862
20,Single-Stranded DNA Binding (GO:0003697),-1.375777,0.094872,0.998,1.0
21,Ubiquitin Binding (GO:0043130),-1.359001,0.102322,0.999,1.0
23,GTPase Regulator Activity (GO:0030695),-1.328511,0.101404,1.0,1.0


## MSigDB_Hallmark_2020

In [11]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='MSigDB_Hallmark_2020', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

2026-02-02 11:09:46,565 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [12]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,p53 Pathway,1.606321,0.002924,0.215,0.471552
2,Estrogen Response Early,1.566862,0.008086,0.284,0.317462
3,Cholesterol Homeostasis,1.501027,0.042184,0.433,0.370063
5,Androgen Response,1.43902,0.051213,0.555,0.410751
10,Allograft Rejection,1.363014,0.043732,0.738,0.535044
11,Estrogen Response Late,1.361683,0.043732,0.74,0.449274
12,Unfolded Protein Response,1.334365,0.142512,0.781,0.440256
13,IL-2/STAT5 Signaling,1.286976,0.059155,0.868,0.505433
15,mTORC1 Signaling,1.235128,0.136483,0.931,0.588099
16,Wnt-beta Catenin Signaling,1.231807,0.207254,0.935,0.536715


In [13]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,G2-M Checkpoint,-1.633574,0.006154,0.136,0.073812
4,Spermatogenesis,-1.468485,0.060914,0.584,0.227791
6,DNA Repair,-1.435023,0.052901,0.678,0.197484
7,Oxidative Phosphorylation,-1.399611,0.067395,0.779,0.204328
8,Mitotic Spindle,-1.38439,0.061934,0.817,0.182331
9,E2F Targets,-1.378906,0.078947,0.831,0.159193
14,Interferon Alpha Response,-1.272851,0.124611,0.967,0.275207
26,Adipogenesis,-1.022899,0.424961,1.0,0.798308
30,Bile Acid Metabolism,-0.923083,0.587931,1.0,0.989322
31,Interferon Gamma Response,-0.839844,0.77259,1.0,1.0


## Cell types

In [14]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='PanglaoDB_Augmented_2021', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

2026-02-02 11:10:02,435 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Gastric Chief Cells,2.220668,0.0,0.0,0.0
1,Salivary Mucous Cells,2.177413,0.0,0.0,0.0
2,Cholangiocytes,2.036495,0.0,0.003,0.001335
3,Mammary Epithelial Cells,2.030373,0.0,0.004,0.001335
4,Keratinocytes,1.999058,0.0,0.011,0.003203
5,Luminal Epithelial Cells,1.982635,0.0,0.016,0.003782
6,Foveolar Cells,1.968645,0.0,0.018,0.003623
8,Microfold Cells,1.925072,0.0,0.023,0.004004
9,Sebocytes,1.910495,0.0,0.026,0.004152
10,Epithelial Cells,1.905622,0.0,0.029,0.004137


In [15]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
7,Pluripotent Stem Cells,-1.965991,0.0,0.002,0.00242
20,Epiblast Cells,-1.723253,0.001567,0.147,0.079066
23,Embryonic Stem Cells,-1.569442,0.002994,0.639,0.357948
24,Gamma Delta T Cells,-1.568614,0.015385,0.641,0.270075
27,Oxyphil Cells,-1.466755,0.054098,0.918,0.547491
30,Adrenergic Neurons,-1.414405,0.055456,0.969,0.700029
31,Ependymal Cells,-1.40723,0.076285,0.972,0.635755
32,Myocytes,-1.403557,0.047544,0.974,0.573631
36,Osteocytes,-1.338558,0.067976,0.996,0.828668
39,Microglia,-1.309889,0.071429,0.999,0.903691


Epithelial markers are strongly enriched among the control drivers

## ChipSeq targets, ChEA

In [16]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ChEA_2022', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-02-02 11:10:14,662 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [17]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
24,RARG 19884340 ChIP-ChIP MEFs Mouse,1.730881,0.0,0.221,0.318778
26,JARID2 20075857 ChIP-Seq MESCs Mouse,1.713658,0.0,0.248,0.183671
29,SUZ12 18974828 ChIP-Seq MESCs Mouse,1.686985,0.0,0.3,0.154408
38,MTF2 20144788 ChIP-Seq MESCs Mouse,1.639123,0.0,0.429,0.183671
42,SUZ12 27294783 Chip-Seq ESCs Mouse,1.612293,0.0,0.503,0.183546
49,SUZ12 20075857 ChIP-Seq MESCs Mouse,1.550556,0.0,0.699,0.267101
51,TP63 17297297 ChIP-ChIP HaCaT Human,1.54484,0.007712,0.72,0.24104
52,PITX1 30713093 ChIP-Seq Epithelial Human Tongu...,1.543706,0.027569,0.724,0.212
58,TP63 30713093 ChIP-Seq Epithelial Human Tongue...,1.507623,0.0,0.819,0.253473
62,JARID2 20064375 ChIP-Seq MESCs Mouse,1.493164,0.0,0.852,0.25689


In [18]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 23109430 ChIP-Seq U2OS Human,-2.069772,0.0,0.0,0.0
1,SMAD1 18555785 ChIP-Seq MESCs Mouse,-2.06917,0.0,0.0,0.0
2,TCF3 18692474 ChIP-Seq MESCs Mouse,-2.031116,0.0,0.0,0.0
3,MYBL2 22936984 ChIP-ChIP MESCs Mouse,-2.028174,0.0,0.0,0.0
4,NACC1 18358816 ChIP-ChIP MESCs Mouse,-2.026743,0.0,0.0,0.0
5,NANOG 18700969 ChIP-ChIP MESCs Mouse,-1.998373,0.0,0.0,0.0
6,NANOG 18555785 ChIP-Seq MESCs Mouse,-1.925797,0.0,0.004,0.000518
7,SOX2 18692474 ChIP-Seq MESCs Mouse,-1.921982,0.0,0.004,0.000454
8,POU5F1 18358816 ChIP-ChIP MESCs Mouse,-1.919393,0.0,0.005,0.000504
9,POU5F1 18700969 ChIP-ChIP MESCs Mouse,-1.909834,0.0,0.006,0.000544


## Enriched targets from ChipSeq, Ensembl

In [19]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ENCODE_TF_ChIP-seq_2015', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-02-02 11:11:13,634 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [20]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,FOSL1 C2C12 mm9,1.96268,0.0,0.019,0.021294
8,SMARCC1 HeLa-S3 hg19,1.723216,0.0,0.208,0.135609
21,ZEB1 HepG2 hg19,1.635744,0.01039,0.412,0.216301
23,TAL1 G1E-ER4 mm9,1.616329,0.0,0.468,0.200051
25,ATF3 K562 hg19,1.570052,0.0,0.611,0.257993
31,EP300 ECC-1 hg19,1.540566,0.003077,0.698,0.279623
42,SMARCC2 HeLa-S3 hg19,1.498257,0.016393,0.799,0.350949
46,MAX myocyte mm9,1.47301,0.009375,0.86,0.381329
48,RAD21 ECC-1 hg19,1.462772,0.010239,0.878,0.370713
52,JUND GM12878 hg19,1.447306,0.033766,0.904,0.381609


In [21]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,E2F4 MEL cell line mm9,-2.037147,0.0,0.0,0.0
2,FOXM1 ECC-1 hg19,-1.840812,0.0,0.028,0.013301
3,IRF3 GM12878 hg19,-1.807172,0.0,0.046,0.015201
4,E2F4 CH12.LX mm9,-1.804058,0.0,0.048,0.011876
5,NANOG H1-hESC hg19,-1.740666,0.001751,0.147,0.033442
6,E2F4 HeLa-S3 hg19,-1.736897,0.0,0.155,0.029926
7,SP2 K562 hg19,-1.735902,0.0,0.158,0.026194
9,NFYA GM12878 hg19,-1.717984,0.0,0.213,0.031945
10,FOS K562 hg19,-1.694223,0.0,0.281,0.040535
11,FOXM1 MCF-7 hg19,-1.691418,0.003205,0.289,0.037812


## TF Perturbation followed by expression

In [11]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=16,
    min_size=5,
    max_size=1000
)

2026-02-06 14:50:11,378 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [12]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(30) # Top Day 12

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
19,prerank,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,0.540459,1.909149,0.0,0.15048,0.125,29/70,15.17%,Lgals7;S100a16;Prr13;Lad1;Dsp;Gsto1;Pkp1;Perp;...
24,prerank,PPARA KO MOUSE GSE6864 CREEDSID GENE 981 UP,0.537065,1.893438,0.0,0.098468,0.164,27/67,13.87%,Ctsz;Prr13;Lad1;Gsto1;Ifi30;Cdkn1a;Krt8;Dgat2;...
25,prerank,HSF1 KO MOUSE GSE41005 CREEDSID GENE 2143 DOWN,0.692002,1.888126,0.0,0.070022,0.174,9/20,5.66%,Lad1;Ccnd2;Fabp5;Pla2g12a;Tuba4a;Ifrd1;Myc;Atf...
32,prerank,PPARA KO MOUSE GSE6864 CREEDSID GENE 1371 DOWN,0.525552,1.856044,0.0,0.082562,0.245,27/68,13.87%,Ctsz;Prr13;Lad1;Gsto1;Ifi30;Cdkn1a;Krt8;Dgat2;...
36,prerank,OVOL2 OE MOUSE GSE55074 CREEDSID GENE 2603 DOWN,0.525539,1.837582,0.0,0.079785,0.294,26/56,14.02%,Lgals7;S100a16;Dsp;Gsto1;Ccnd2;Pkp1;Perp;Fabp5...
37,prerank,PPARA DEFICIENCY MOUSE GSE6864 CREEDSID GENE 6...,0.517132,1.823507,0.0,0.076755,0.324,26/67,13.87%,Ctsz;Prr13;Lad1;Dsp;Gsto1;Krt8;Sat1;Tuba4a;Mis...
42,prerank,MYC KD HUMAN GSE22139 CREEDSID GENE 712 DOWN,0.632107,1.799886,0.0,0.085411,0.402,13/22,13.92%,Ccnd2;Fabp5;Ifrd1;Myc;Tmem147;Zfas1;Snhg1;Ppa1...
46,prerank,YY1 KD MOUSE GSE31784 CREEDSID GENE 1333 UP,0.551954,1.78812,0.0,0.085213,0.447,16/40,14.02%,Lad1;Gsto1;Ly6g6c;Dgat2;Tubb3;Gpx2;Gatm;Pcolce...
50,prerank,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,0.528913,1.771201,0.0,0.089211,0.496,13/50,6.06%,Lgals7;Dsp;Mal2;Perp;Ly6g6c;Ovol1;Anxa8;Krt17;...
53,prerank,PPARG OE MOUSE GSE2192 CREEDSID GENE 863 UP,0.528059,1.765647,0.002994,0.084733,0.512,21/48,13.82%,Nupr1;Fabp5;Pla2g12a;Cdkn1a;Mgst3;Tmem147;Tob1...


Many proteins related to Ovol1 appear, including Myc, Ovol2 and Ovol1 itself. Directions are inconsistent.

In [5]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 1660 DOWN,-2.450379,0.0,0.0,0.0
1,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 2289 DOWN,-2.450205,0.0,0.0,0.0
2,MTF2 KD MOUSE GSE16364 CREEDSID GENE 1355 UP,-2.41091,0.0,0.0,0.0
3,ZFP42 KD MOUSE GSE9978 CREEDSID GENE 1334 UP,-2.360429,0.0,0.0,0.0
4,ZFX KO MOUSE GSE7069 CREEDSID GENE 177 DOWN,-2.248554,0.0,0.0,0.0
5,ZFP42 KD MOUSE GSE9978 CREEDSID GENE 1335 UP,-2.243119,0.0,0.0,0.0
6,MSGN1 OE MOUSE GSE29848 CREEDSID GENE 1629 DOWN,-2.189715,0.0,0.001,0.000142
7,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 2585 DOWN,-2.117429,0.0,0.002,0.000249
8,FOXD3 OE MOUSE GSE58960 CREEDSID GENE 1415 DOWN,-2.116637,0.0,0.002,0.000221
9,SOX2 KD MOUSE GSE39771 CREEDSID GENE 1757 DOWN,-2.112671,0.0,0.002,0.000199


## Test if specific gene sets are enriched

In [33]:
# Get libraries

panglao_lib = gseapy.get_library(name='PanglaoDB_Augmented_2021', organism='Mouse')
go_lib = gseapy.get_library(name='GO_Biological_Process_2025', organism='Mouse')
go_molecular_function = gseapy.get_library(name='GO_Molecular_Function_2025', organism='Mouse')
hallmark_lib = gseapy.get_library(name='MSigDB_Hallmark_2020', organism='Mouse')

In [34]:
target_sets = {
    "Apoptosis Activation": go_lib["Positive Regulation of Apoptotic Process (GO:0043065)"],
    "Apoptosis Inhibition": go_lib["Negative Regulation of Apoptotic Process (GO:0043066)"],
    "Cell Cycle Inhibition": go_lib["Negative Regulation of Cell Cycle (GO:0045786)"],
    "Cell Cycle Activation": go_lib["Positive Regulation of Cell Cycle (GO:0045787)"],
    "Apoptosis Signaling Inhibition": go_lib["Negative Regulation of Apoptotic Signaling Pathway (GO:2001234)"],
    "Apoptosis Signaling Activation": go_lib["Positive Regulation of Apoptotic Signaling Pathway (GO:2001235)"],
    "Stem Cell Proliferation": go_lib["Positive Regulation of Stem Cell Proliferation (GO:2000648)"],
    "Stem Cell Inhibited Proliferation": go_lib["Negative Regulation of Stem Cell Population Maintenance (GO:1902455)"],
    "Stem Cell Inhibited Differentiation": go_lib["Negative Regulation of Stem Cell Differentiation (GO:2000737)"],
    "Stem Cell Differentiation": go_lib["Positive Regulation of Stem Cell Differentiation (GO:2000738)"],
    "DNA Damage Response": go_lib["DNA Damage Response (GO:0006974)"],
    "DNA_Repair": hallmark_lib['DNA Repair'],
    "P53_Pathway": hallmark_lib['p53 Pathway'],
    "Keratinocytes": panglao_lib["Keratinocytes"],
    "Epithelial cells": panglao_lib["Epithelial Cells"]
}

rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets=target_sets,
    threads=16,
    min_size=5,
    max_size=2000,
    permutation_num=1000,
    seed=0
)

pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-02-02 11:19:48,524 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Keratinocytes,0.624645,2.0251,0.0,0.001222,0.001,22/41,17.03%,Lgals7;Lad1;Pkp1;Perp;Ovol1;Anxa8;S100a14;Krt1...
1,prerank,Epithelial cells,0.541295,1.924159,0.0,0.001222,0.002,32/67,19.08%,Lad1;Crb3;Mal2;Ovol1;Krt8;Dgat2;Gpx2;Misp;S100...
2,prerank,P53_Pathway,0.465473,1.60209,0.003289,0.032179,0.078,24/60,18.48%,Ccnd2;Ifi30;Nupr1;Perp;Cdkn1a;Sat1;Gpx2;Krt17;...
3,prerank,Apoptosis Signaling Activation,0.654014,1.540293,0.045024,0.038187,0.118,5/11,16.88%,Nupr1;Bbc3;App;Tnfrsf12a;Ddit3
7,prerank,Apoptosis Inhibition,0.210775,0.79448,0.944444,0.966599,0.978,18/107,15.12%,Ccnd2;Nupr1;Prkaa2;Cd44;Myc;Anxa5;Tmbim1;Ddah2...
10,prerank,Cell Cycle Inhibition,0.281041,0.694447,0.905,0.916497,0.988,1/14,1.40%,Nupr1
12,prerank,Apoptosis Signaling Inhibition,-0.238535,-0.560375,0.964346,0.984178,1.0,4/14,33.20%,Tcf7l2;Bmp4;Thbs1;Clu
11,prerank,Stem Cell Proliferation,-0.283705,-0.572438,0.962838,1.0,1.0,1/8,7.46%,Nanog
9,prerank,Cell Cycle Activation,-0.29015,-0.707803,0.871545,1.0,1.0,2/16,4.96%,Tcf7l1;Cited2
8,prerank,Apoptosis Activation,-0.241989,-0.775089,0.896603,1.0,1.0,29/63,37.66%,Dlc1;Rest;Gadd45b;Top2a;Bnip3;Tsc22d1;Sfrp1;Ec...


# Differential expression analysis with Scanpy

Scanpy uses a Wilcoxon rank sum test.

In [5]:
# Assign each cell to the most likely macrostate

macrostates = adata.uns["coarse_fwd"].obs["coarse_init_dist"].index.to_list()
macrostate_assignment = [macrostates[numpy.argmax(macrostate_probabilities)] for macrostate_probabilities in adata.obsm["macrostates_fwd_memberships"]]
adata.obs["macrostate"] = macrostate_assignment
adata.uns["macrostate_colors"] = [
    adata.uns["macrostates_fwd_colors"][5],
    adata.uns["macrostates_fwd_colors"][2],
    adata.uns["macrostates_fwd_colors"][3],
    adata.uns["macrostates_fwd_colors"][4],
    adata.uns["macrostates_fwd_colors"][1],
    adata.uns["macrostates_fwd_colors"][0],
    adata.uns["macrostates_fwd_colors"][6]
]

## Comparison between all macrostates

In [13]:
# perform differential expression using the standard wilcoxon rank-sum test
scanpy.tl.rank_genes_groups(
    adata,
    groupby="macrostate",
    method="wilcoxon",
    use_raw=False,
    key_added="macrostate_rank_genes"
)

# extract the results into a dataframe for a specific target macrostate
dead_end_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 12 Control"
)
dead_end_rank_df.index = dead_end_rank_df["names"]
dead_end_rank_df = dead_end_rank_df.drop(columns="names")

control_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 4 Control"
)
control_rank_df.index = control_rank_df["names"]
control_rank_df = control_rank_df.drop(columns="names")

In [36]:
dead_end_rank_df.head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Zfas1,26.189415,2.912675,3.507877e-151,7.015755e-148
2410006H16Rik,25.884188,2.383039,1.003519e-147,1.003519e-144
Snhg1,24.932861,2.789920,3.276535e-137,2.184357e-134
1110038B12Rik,24.600237,2.454037,1.255827e-133,6.279133e-131
Snhg12,24.165182,3.138035,5.170944e-129,2.068378e-126
Gas5,23.397280,1.672911,4.554957e-121,1.518319e-118
Ifrd1,22.545397,2.568011,1.490053e-112,4.257295e-110
Cdkn1a,21.841169,2.205297,9.431106e-106,2.357776e-103
Ppp1r15a,21.520761,2.846743,9.951511e-103,2.211447e-100


In [37]:
# Most upregulated transcription factors in dead end

dead_end_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Atf3,18.789471,4.924244,9.209480e-79,8.372254e-77
Maff,14.781541,3.392600,1.927055e-49,5.416949e-48
Ddit3,13.226619,2.166627,6.159631e-40,1.140672e-38
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Klf10,12.000822,1.918085,3.517865e-33,4.753872e-32
Arid3a,10.603693,1.693256,2.864403e-26,2.822072e-25
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Junb,10.341942,1.311702,4.552173e-25,4.234580e-24


Really exciting! The top three TFs are involved in stress response, suggesting that this macrostate corresponds to high activity of 

In [48]:
hic2_target_scores.loc["Klf10"]

chip_score    4.637256
Name: Klf10, dtype: float64

In [44]:
# Most upregulated transcription factors in dead end that are shared targets

dead_end_rank_df.loc[numpy.intersect1d(target_tfs, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Ddit3,13.226619,2.166627,6.159631e-40,1.140672e-38
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Klf10,12.000822,1.918085,3.517865e-33,4.753872e-32
Arid3a,10.603693,1.693256,2.864403e-26,2.822072e-25
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Junb,10.341942,1.311702,4.552173e-25,4.234580e-24
Klf4,8.294026,1.242020,1.094780e-16,6.185198e-16
Phox2a,7.614943,2.153333,2.638079e-14,1.253244e-13


In [38]:
# Most downregulated transcription factors in dead end

dead_end_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=True).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Cenpa,-14.694151,-2.029778,7.027420e-49,1.873979e-47
Hmgb2,-12.436042,-1.655495,1.665552e-35,2.485899e-34
Tcf7l1,-10.125731,-1.699106,4.247917e-24,3.646281e-23
Prrx1,-9.667671,-3.502777,4.136854e-22,3.182196e-21
Jarid2,-9.106974,-1.539088,8.471185e-20,5.782379e-19
Terf1,-8.995116,-1.125859,2.359814e-19,1.578471e-18
Nfib,-8.488363,-2.268551,2.095683e-17,1.240049e-16
Prrx2,-8.243245,-5.525078,1.676023e-16,9.337178e-16
Zeb1,-8.197787,-3.580868,2.448527e-16,1.345345e-15


In [39]:
# Most upregulated transcription factors in day 4 control

control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Tlx2,56.619308,4.156113,0.000000e+00,0.000000e+00
Dmrtc2,47.219395,3.756907,0.000000e+00,0.000000e+00
Zbtb7c,43.116035,2.244280,0.000000e+00,0.000000e+00
Hmgb3,39.926929,1.578542,0.000000e+00,0.000000e+00
Myc,39.168621,1.562202,0.000000e+00,0.000000e+00
Ovol1,35.878239,3.140644,6.674091e-282,1.026783e-280
Zbtb32,35.045231,2.059280,4.609327e-269,6.778422e-268
Hoxa9,28.716011,1.644503,2.407959e-181,2.239962e-180
Phox2a,28.670206,2.174598,8.977100e-181,8.273825e-180


In [25]:
dead_end_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Klf4,8.294026,1.242020,1.094780e-16,6.185198e-16
Tlx2,4.417089,0.874272,1.000390e-05,2.513543e-05
Hic2,-1.219937,-0.087063,2.224888e-01,3.171616e-01
Rest,-5.134597,-0.644595,2.827485e-07,8.124958e-07
Zfp42,-7.309782,-1.655304,2.675775e-13,1.197215e-12
Jarid2,-9.106974,-1.539088,8.471185e-20,5.782379e-19


In [23]:
# Cell types in the dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:11,075 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Mammary Epithelial Cells,2.588875,0.0,0.0,0.0
1,Trichocytes,2.454958,0.0,0.0,0.0
2,Keratinocytes,2.450678,0.0,0.0,0.0
3,Luminal Epithelial Cells,2.435519,0.0,0.0,0.0
4,Gastric Chief Cells,2.295272,0.0,0.0,0.0
5,Principal Cells,2.268925,0.0,0.0,0.0
6,Cholangiocytes,2.253806,0.0,0.0,0.0
7,Salivary Mucous Cells,2.175057,0.0,0.0,0.0
8,Sebocytes,2.173433,0.0,0.0,0.0
9,Airway Goblet Cells,2.135118,0.0,0.0,0.0


In [26]:
# MSig Hallmarks in dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:47,611 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,p53 Pathway,2.27238,0.0,0.0,0.0
3,TNF-alpha Signaling via NF-kB,1.981754,0.0,0.003,0.002193
6,Estrogen Response Early,1.623119,0.0,0.2,0.107212
7,UV Response Up,1.613178,0.005952,0.213,0.087354
8,mTORC1 Signaling,1.591396,0.006135,0.24,0.082164
9,Hypoxia,1.571308,0.0,0.268,0.079678
12,Androgen Response,1.432537,0.041056,0.565,0.18066
14,KRAS Signaling Dn,1.3821,0.082621,0.671,0.221126
17,Estrogen Response Late,1.338374,0.04893,0.782,0.254548
18,Unfolded Protein Response,1.335315,0.135135,0.788,0.232895


In [27]:
# MSig Hallmarks in day 4 control

pre_res = gseapy.prerank(
    rnk=control_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:48,469 [WARNING] Duplicated values found in preranked stats: 0.55% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Estrogen Response Late,1.650953,0.001751,0.15,0.162436
2,Estrogen Response Early,1.568822,0.010363,0.318,0.192184
4,Myc Targets V1,1.406796,0.09589,0.753,0.486363
5,E2F Targets,1.397855,0.062278,0.771,0.390271
7,p53 Pathway,1.348818,0.069027,0.861,0.429322
8,Wnt-beta Catenin Signaling,1.346102,0.142562,0.865,0.364536
9,mTORC1 Signaling,1.345051,0.075134,0.865,0.314888
10,Notch Signaling,1.342163,0.138493,0.87,0.278478
11,Xenobiotic Metabolism,1.282317,0.109319,0.947,0.353098
12,Androgen Response,1.269339,0.15283,0.958,0.341587


In [28]:
# GO biological process in dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="GO_Biological_Process_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:49,209 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Positive Regulation of Intrinsic Apoptotic Sig...,1.760335,0.007937,0.85,1.0
5,Regulation of Autophagy (GO:0010506),1.720115,0.004975,0.953,1.0
10,Antigen Processing and Presentation of Exogeno...,1.702887,0.007895,0.971,0.841609
11,Antigen Processing and Presentation of Exogeno...,1.702887,0.007895,0.971,0.841609
9,Antigen Processing and Presentation of Peptide...,1.702887,0.007895,0.971,0.841609
15,Positive Regulation of Sprouting Angiogenesis ...,1.646528,0.023018,0.996,1.0
18,Intracellular Iron Ion Homeostasis (GO:0006879),1.634716,0.026247,0.999,1.0
19,Regulation of Glycolytic Process (GO:0006110),1.62813,0.028796,0.999,1.0
20,Antigen Processing and Presentation of Peptide...,1.623347,0.018182,0.999,1.0
24,Positive Regulation of Neuron Apoptotic Proces...,1.609309,0.039578,1.0,1.0


In [29]:
# TFs whose targets are upregulated in dead end
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:54,082 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
108,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,1.967528,0.0,0.056,0.042759
142,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,1.914332,0.0,0.12,0.045275
143,PLAGL2 DEFICIENCY MOUSE GSE9123 CREEDSID GENE ...,1.913938,0.0,0.12,0.030393
154,ARID3A KD MOUSE GSE56853 CREEDSID GENE 1337 UP,1.90636,0.0,0.135,0.025467
159,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1512 DOWN,1.896851,0.0,0.156,0.019598
158,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1511 UP,1.896851,0.0,0.156,0.019598
171,PPARA KO MOUSE GSE6864 CREEDSID GENE 981 UP,1.874495,0.0,0.217,0.023626
176,FOXO1 KD MOUSE GSE6623 CREEDSID GENE 505 DOWN,1.870261,0.0,0.23,0.022166
187,GLI1 INHIBITION HUMAN GSE36855 CREEDSID GENE 6...,1.858362,0.0,0.257,0.022498
200,PPARA KO MOUSE GSE6864 CREEDSID GENE 1371 DOWN,1.840598,0.0,0.308,0.025027


In [30]:
# TFs whose targets are downregulated in dead end
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:45:58,331 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXA1 KD HUMAN GSE25315 CREEDSID GENE 2164 DOWN,-2.463419,0.0,0.0,0.0
1,OVOL2 OE PC3 HUMAN GSE48230 RNASEQ DOWN,-2.452743,0.0,0.0,0.0
2,DNMT1 KO MOUSE GSE31626 CREEDSID GENE 1237 DOWN,-2.417583,0.0,0.0,0.0
3,E2F1 OE HUMAN GSE2715 CREEDSID GENE 1129 DOWN,-2.407181,0.0,0.0,0.0
4,TAL1 KD JURKAT HUMAN GSE72299 RNASEQ DOWN,-2.372448,0.0,0.0,0.0
5,MYB KD HUMAN GSE49286 CREEDSID GENE 1842 DOWN,-2.353719,0.0,0.0,0.0
6,MEIS2 KD KASUMI1 HUMAN GSE81328 RNASEQ UP,-2.337821,0.0,0.0,0.0
7,ZNF750 KD HUMAN GSE38039 CREEDSID GENE 204 UP,-2.336913,0.0,0.0,0.0
8,ZNF750 KD HUMAN GSE38039 CREEDSID GENE 2704 DOWN,-2.336913,0.0,0.0,0.0
9,ZMAT4 SIRNA T47D HUMAN GSE79586 RNASEQ DOWN,-2.332155,0.0,0.0,0.0


In [31]:
# TFs whose targets are upregulated in control
pre_res = gseapy.prerank(
    rnk=control_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:46:05,247 [WARNING] Duplicated values found in preranked stats: 0.55% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
7,OVOL2 OE MOUSE GSE55074 CREEDSID GENE 2603 DOWN,2.05011,0.0,0.005,0.003324
9,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,2.039413,0.0,0.007,0.002327
10,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,2.039194,0.0,0.007,0.001551
38,HSF1 KO MOUSE GSE41005 CREEDSID GENE 2143 DOWN,1.856263,0.0,0.234,0.048202
39,YY1 KD MOUSE GSE31784 CREEDSID GENE 1333 UP,1.850319,0.0,0.257,0.042418
56,MBD3 KD MOUSE GSE31008 CREEDSID GENE 1325 UP,1.805803,0.001669,0.429,0.070475
62,PPARA DEFICIENCY MOUSE GSE6864 CREEDSID GENE 6...,1.793981,0.0,0.474,0.06943
63,KLF1 KO MOUSE GSE36427 CREEDSID GENE 1562 DOWN,1.790153,0.001855,0.496,0.06499
65,NFE2L2 KO MOUSE GSE18344 CREEDSID GENE 965 DOWN,1.785133,0.0,0.512,0.061093
78,HNF4A MUT MOUSE GSE3116 CREEDSID GENE 850 DOWN,1.763923,0.0,0.604,0.073135


These results suggest that the control pathway has low Ovol2 activity, which is consistent with high Ovol1 activity as Ovol1 downregulates Ovol2.

## Compare day 12 control with day 4 control

In [40]:
# Compare day 12 control with day 4 control
scanpy.tl.rank_genes_groups(
    adata,
    groupby="macrostate",
    method="wilcoxon",
    use_raw=False,
    reference="Day 4 Control",
    key_added="macrostate_vs_control_rank_genes"
)
dead_end_vs_control_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    group="Day 12 Control",
    key="macrostate_vs_control_rank_genes"
)
dead_end_vs_control_rank_df.index = dead_end_vs_control_rank_df["names"]
dead_end_vs_control_rank_df = dead_end_vs_control_rank_df.drop(columns="names")
dead_end_vs_control_rank_df.head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
2410006H16Rik,26.126083,2.549994,1.843133e-150,3.686265e-147
Zfas1,25.591373,2.887306,1.903077e-144,1.903077e-141
1110038B12Rik,25.104904,2.706954,4.396385e-139,2.930924e-136
Snhg1,24.437115,2.785392,6.898948e-132,3.449474e-129
Gas5,24.352331,1.764712,5.476893e-131,2.190757e-128
Snhg12,24.150040,3.244454,7.459324e-129,2.486441e-126
Ppp1r15a,21.102528,2.884851,7.539659e-99,1.507932e-96
Sqstm1,19.848444,2.438090,1.136654e-87,1.623791e-85
Cstb,19.777744,1.662232,4.629430e-87,6.172574e-85


In [41]:
# Most upregulated transcription factors in dead end

dead_end_vs_control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Atf3,17.293951,4.182650,5.224525e-67,3.483017e-65
Ddit3,15.975071,3.221851,1.906363e-57,9.776220e-56
Maff,14.538376,3.518187,6.921518e-48,2.307173e-46
Crebrf,11.278059,3.426975,1.684214e-29,2.716475e-28
Zbtb10,9.688887,2.093428,3.361729e-22,3.954976e-21
Junb,9.540514,1.198861,1.421259e-21,1.615067e-20
Arid3a,9.539727,1.519866,1.432088e-21,1.618178e-20
Klf6,9.515185,1.901875,1.813887e-21,2.026689e-20
Klf4,9.149061,1.463191,5.743040e-20,5.771900e-19


Top 4 TFs are all involved in stress responses

In [42]:
# Most downregulated transcription factors in dead end

dead_end_vs_control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=True).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Cenpa,-18.378662,-2.361419,1.947120e-75,1.557696e-73
Hmgb3,-14.654617,-2.041724,1.258640e-48,4.266575e-47
Hmgb2,-11.795198,-1.436325,4.132265e-32,7.313744e-31
Id2,-8.479613,-1.096260,2.259429e-17,1.990686e-16
Tlx2,-8.452542,-1.106642,2.850273e-17,2.500239e-16
Prrx1,-8.317663,-2.595609,8.971450e-17,7.700815e-16
Mis18bp1,-7.335813,-1.151761,2.203806e-13,1.541123e-12
Terf1,-7.233290,-0.849773,4.714325e-13,3.217969e-12
Zfhx4,-6.941576,-1.290629,3.877483e-12,2.493558e-11


In [35]:
dead_end_vs_control_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Klf4,9.149061,1.463191,5.743040e-20,5.771900e-19
Ovol1,6.053084,1.239698,1.420988e-09,7.722761e-09
Myc,5.996467,0.733084,2.016562e-09,1.084173e-08
Rest,3.244519,0.294803,1.176493e-03,3.435017e-03
Zbtb7c,3.167943,0.454898,1.535217e-03,4.405213e-03
Hic2,1.094972,0.462989,2.735291e-01,4.687730e-01
Tcf7l1,-4.097703,-0.527595,4.172700e-05,1.498276e-04
Zfp42,-4.107964,-0.588239,3.991626e-05,1.441020e-04
Jarid2,-4.346265,-0.551580,1.384752e-05,5.255225e-05


The dead end state is signified by high expression of Klf4, Myc and Ovol1.

Tlx2 is downregulated in the dead end, suggesting it does not bring cells into the dead end. It could just be correlated with the control pathway.

In [ ]:
# Cell types in the dead end

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]],
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:31:58,986 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Mammary Epithelial Cells,2.032868,0.0,0.0,0.0
1,Principal Cells,1.998692,0.0,0.0,0.0
2,Luminal Epithelial Cells,1.976177,0.0,0.003,0.001648
3,Trichocytes,1.862625,0.003597,0.028,0.010508
4,Gastric Chief Cells,1.796434,0.0,0.062,0.018296
6,Sebocytes,1.740809,0.009375,0.12,0.030699
8,Cholangiocytes,1.672694,0.006515,0.229,0.05563
9,Keratinocytes,1.635431,0.003597,0.303,0.071855
10,Airway Epithelial Cells,1.616102,0.02029,0.363,0.079118
11,Nuocytes,1.613408,0.035294,0.37,0.072689


In [ ]:
# MSig Hallmarks

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:32:04,161 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,TNF-alpha Signaling via NF-kB,2.23291,0.0,0.0,0.0
1,p53 Pathway,2.080421,0.0,0.0,0.0
4,Hypoxia,1.837935,0.0,0.01,0.005158
7,UV Response Up,1.595374,0.0,0.175,0.075826
11,Unfolded Protein Response,1.436948,0.081301,0.477,0.21974
13,mTORC1 Signaling,1.398681,0.035484,0.576,0.239599
16,TGF-beta Signaling,1.300711,0.137931,0.792,0.37095
20,Estrogen Response Early,1.19468,0.152941,0.928,0.550124
21,Protein Secretion,1.17724,0.271186,0.936,0.531985
22,IL-6/JAK/STAT3 Signaling,1.161149,0.194529,0.945,0.512521


In [ ]:
# GO biological process

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="GO_Biological_Process_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:32:26,962 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Regulation of JNK Cascade (GO:0046328),1.805935,0.0,0.323,0.486163
1,Response to Endoplasmic Reticulum Stress (GO:0...,1.799198,0.003125,0.351,0.273211
2,Positive Regulation of JNK Cascade (GO:0046330),1.792325,0.0,0.382,0.202908
6,Macroautophagy (GO:0016236),1.670805,0.010929,0.913,0.949601
7,Anterograde Trans-Synaptic Signaling (GO:0098916),1.66677,0.012698,0.922,0.800535
8,Regulation of Translation (GO:0006417),1.651243,0.012012,0.959,0.820315
9,Positive Regulation of Granulocyte Chemotaxis ...,1.648447,0.010724,0.964,0.732454
10,Integrated Stress Response Signaling (GO:0140467),1.644288,0.002681,0.966,0.674985
13,Immunoglobulin Mediated Immune Response (GO:00...,1.632127,0.005602,0.977,0.697809
17,T Cell Chemotaxis (GO:0010818),1.615536,0.010724,0.984,0.766932


All the top terms are related to damage and stress signaling.

In [ ]:
# ENCODE ChIP-Seq
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="ENCODE_TF_ChIP-seq_2015",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:35:59,059 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,FOSL2 HepG2 hg19,1.845511,0.0,0.032,0.049575
5,MAX myocyte mm9,1.794046,0.0,0.061,0.048158
6,PML MCF-7 hg19,1.772045,0.003846,0.083,0.042493
7,NR2F2 MCF-7 hg19,1.756168,0.0,0.096,0.038952
10,TAF1 MCF-7 hg19,1.739986,0.0,0.117,0.039093
11,NR3C1 ECC-1 hg19,1.739029,0.003484,0.117,0.032814
14,NFE2 K562 hg19,1.696168,0.009009,0.193,0.046944
15,USF1 K562 hg19,1.691845,0.0,0.198,0.042316
16,YY1 GM12891 hg19,1.6876,0.007117,0.21,0.039975
17,MEF2A K562 hg19,1.660891,0.003185,0.265,0.047308


In [ ]:
# Chsea
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="ChEA_2022",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:35:08,388 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,FOXA1 33576154 ChIP-Seq Human HilarLN Prostate...,1.891787,0.0,0.013,0.01714
4,RELB 30642670 ChIP-Seq CTB1 Human Placenta Inf...,1.788053,0.003236,0.068,0.046145
6,TP63 17297297 ChIP-ChIP HaCaT Human,1.713096,0.003115,0.151,0.073832
8,NOTCH1 21737748 ChIP-Seq TLL Human,1.6068,0.018237,0.396,0.177659
9,PKCTHETA 26484144 Chip-Seq BREAST Human,1.601674,0.003676,0.414,0.150038
10,FOXA1 26457646 ChIP-Seq LHSAR Human ProstateCa...,1.591983,0.0,0.446,0.137337
11,FOXO1 32281255 ChIP-Seq Chondrocytes Human Ost...,1.590245,0.0,0.451,0.119036
12,MEIS1 20887958 ChIP-Seq HPC-7 Mouse,1.577726,0.0,0.489,0.116681
13,NUCKS1 24931609 ChIP-Seq HEPATOCYTES Mouse,1.572461,0.0,0.506,0.10899
14,BRD4 28847988 ChIP-Seq BCBL1 Human Blood Lymphoma,1.561826,0.019231,0.54,0.109166


In [ ]:
# TF perturbation
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-05 09:35:28,265 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,GLI1 INHIBITION HUMAN GSE36855 CREEDSID GENE 6...,2.180205,0.0,0.0,0.0
20,FOXO1 KD MOUSE GSE6623 CREEDSID GENE 505 DOWN,1.988647,0.0,0.016,0.00618
29,SOX4 KD HUMAN GSE4225 CREEDSID GENE 64 UP,1.91645,0.0,0.061,0.016738
45,NFKB1 INACTIVATION HUMAN GSE20667 CREEDSID GEN...,1.857504,0.003115,0.132,0.02839
54,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1512 DOWN,1.825931,0.0,0.204,0.031029
55,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1511 UP,1.825931,0.0,0.204,0.031029
56,ZXDC DEPLET MOUSE GSE45417 CREEDSID GENE 1257 UP,1.819913,0.0,0.216,0.028693
62,PLAGL2 DEFICIENCY MOUSE GSE9123 CREEDSID GENE ...,1.806016,0.0,0.249,0.029838
79,CEBPA KO MOUSE GSE61468 CREEDSID GENE 1476 UP,1.775358,0.0,0.362,0.041458
82,ARID3B KO MOUSE GSE62069 CREEDSID GENE 2591 UP,1.772634,0.0,0.372,0.038703
